---
title: CONUS 404 diagnostic plots
author: Harsha R. Hampapura
date: 09-02-2026
---

### Data Access

- This notebook illustrates how to make diagnostic plots using the CONUS 404 dataset hosted on NCAR's Geoscience Data Exchange (GDEX).
- https://gdex.ucar.edu/datasets/d559000/
- This data is open access and can be accessed via 3 protocols
  1) POSIX (if you have access to NCAR's HPC systems: Casper or Derecho)
  2) HTTPS
  3) OSDF using intake-ESM catalogs.
- Learn about intake-ESM catalogs: https://intake-esm.readthedocs.io/en/stable/ 

In [3]:
# Imports 
import intake
import numpy as np
import pandas as pd
import xarray as xr
import seaborn as sns
import matplotlib.pyplot as plt
import os

In [4]:
import dask 
from dask_jobqueue import PBSCluster
from dask.distributed import Client

In [5]:
# Catalog URLs
cat_url     = 'https://data.gdex.ucar.edu/d559000/catalogs/d559000-posix.json' # POSIX access
# cat_url     = 'https://osdf-director.osg-htc.org/ncar/gdex/d559000/catalogs/d559000-osdf.json'
print(cat_url)

https://data.gdex.ucar.edu/d559000/catalogs/d559000-posix.json


In [6]:
# Set up your scratch folder path
username       = os.environ["USER"]
glade_scratch  = "/glade/derecho/scratch/" + username
print(glade_scratch)

/glade/derecho/scratch/harshah


## Create a PBS cluster

In [7]:
# Create a PBS cluster object
cluster = PBSCluster(
    job_name = 'dask-wk25-hpc',
    cores = 1,
    memory = '10GiB',
    processes = 1,
    local_directory = glade_scratch+'/dask/spill/',
    log_directory = glade_scratch + '/dask/logs/',
    resource_spec = 'select=1:ncpus=1:mem=10GB',
    queue = 'casper',
    walltime = '5:00:00',
    #interface = 'ib0'
    interface = 'ext'
)

/glade/u/home/harshah/.conda/envs/osdf/lib/python3.11/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 34393 instead
  warnings.warn(
2026-02-23 16:11:58,638 - tornado.application - ERROR - Uncaught exception GET /status/ws (127.0.0.1)
HTTPServerRequest(protocol='http', host='jupyterhub.hpc.ucar.edu', method='GET', uri='/status/ws', version='HTTP/1.1', remote_ip='127.0.0.1')
Traceback (most recent call last):
  File "/glade/u/home/harshah/.conda/envs/osdf/lib/python3.11/site-packages/tornado/websocket.py", line 965, in _accept_connection
    open_result = handler.open(*handler.open_args, **handler.open_kwargs)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/glade/u/home/harshah/.conda/envs/osdf/lib/python3.11/site-packages/tornado/web.py", line 3388, in wrapper
    return method(self, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  F

In [8]:
# Scale the cluster and display cluster dashboard URL
n_workers = 4
client = Client(cluster)
cluster.scale(n_workers)
client.wait_for_workers(n_workers = n_workers)
cluster

PBSCluster(cea7a354, 'tcp://128.117.208.101:37303', workers=4, threads=4, memory=40.00 GiB)

## Load CONUS 404 data from GDEX using an intake catalog

In [9]:
col = intake.open_esm_datastore(cat_url)
col

,unique
path,79
variable,206
format,1
short_name,206
long_name,119
units,33
start_time,41
end_time,41
level,1
level_units,1


- col.df turns the catalog object into a pandas dataframe!
- (Actually, it accesses the dataframe attribute of the catalog)

In [10]:
col.df

,path,variable,format,short_name,long_name,units,start_time,end_time,level,level_units,frequency
0,/glade/campaign/collections/rda/data/d559000/k...,ACDEWC,reference,ACDEWC,"Accumulated canopy dew rate, accumulated over ...",mm,1979-10-01,1980-09-30 23:00:00,<NA>,<NA>,0 days 01:00:00
1,/glade/campaign/collections/rda/data/d559000/k...,ACDRIPR,reference,ACDRIPR,"Accumulated canopy precipitation drip rate, ac...",mm,1979-10-01,1980-09-30 23:00:00,<NA>,<NA>,0 days 01:00:00
2,/glade/campaign/collections/rda/data/d559000/k...,ACDRIPS,reference,ACDRIPS,"Accumulated canopy snow drip rate, accumulated...",mm,1979-10-01,1980-09-30 23:00:00,<NA>,<NA>,0 days 01:00:00
3,/glade/campaign/collections/rda/data/d559000/k...,ACECAN,reference,ACECAN,Accumulated net evaporation of canopy water (e...,mm,1979-10-01,1980-09-30 23:00:00,<NA>,<NA>,0 days 01:00:00
4,/glade/campaign/collections/rda/data/d559000/k...,ACEDIR,reference,ACEDIR,Accumulated net soil evaporation or snowpack s...,mm,1979-10-01,1980-09-30 23:00:00,<NA>,<NA>,0 days 01:00:00
...,...,...,...,...,...,...,...,...,...,...,...
8574,/glade/campaign/collections/rda/data/d559000/k...,V,reference,V,<NA>,m s-1,2020-10-01,2021-09-30 23:00:00,<NA>,<NA>,0 days 01:00:00
8575,/glade/campaign/collections/rda/data/d559000/k...,W,reference,W,<NA>,m s-1,2020-10-01,2021-09-30 23:00:00,<NA>,<NA>,0 days 01:00:00
8576,/glade/campaign/collections/rda/data/d559000/k...,Z,reference,Z,<NA>,m2 s-2,2020-10-01,2021-09-30 23:00:00,<NA>,<NA>,0 days 01:00:00
8577,/glade/campaign/collections/rda/data/d559000/k...,ilev,reference,ilev,vertical stagger levels,Dimensionless,2020-10-01,2021-09-30 23:00:00,<NA>,<NA>,0 days 01:00:00


## Select data and plot

#### What if you don't know the variable names ?
- Use pandas logic to print out the short_name and long_name

In [11]:
col.df[['variable','long_name']]

,variable,long_name
0,ACDEWC,"Accumulated canopy dew rate, accumulated over ..."
1,ACDRIPR,"Accumulated canopy precipitation drip rate, ac..."
2,ACDRIPS,"Accumulated canopy snow drip rate, accumulated..."
3,ACECAN,Accumulated net evaporation of canopy water (e...
4,ACEDIR,Accumulated net soil evaporation or snowpack s...
...,...,...
8574,V,<NA>
8575,W,<NA>
8576,Z,<NA>
8577,ilev,vertical stagger levels


- We notice that long_name is not available for some variables like 'V'
- In such cases, please look at the wrfout_datadictionary file on this page https://gdex.ucar.edu/datasets/d559000/documentation/#

### Temperature
- Plot temperature for a random date

In [12]:
cat_temp = col.search(variable='T2')
cat_temp.df.head()

,path,variable,format,short_name,long_name,units,start_time,end_time,level,level_units,frequency
0,/glade/campaign/collections/rda/data/d559000/k...,T2,reference,T2,<NA>,K,1979-10-01,1980-09-30 23:00:00,<NA>,<NA>,0 days 01:00:00
1,/glade/campaign/collections/rda/data/d559000/k...,T2,reference,T2,<NA>,K,1980-10-01,1981-09-30 23:00:00,<NA>,<NA>,0 days 01:00:00
2,/glade/campaign/collections/rda/data/d559000/k...,T2,reference,T2,<NA>,K,1981-10-01,1982-09-30 23:00:00,<NA>,<NA>,0 days 01:00:00
3,/glade/campaign/collections/rda/data/d559000/k...,T2,reference,T2,<NA>,K,1982-10-01,1983-09-30 23:00:00,<NA>,<NA>,0 days 01:00:00
4,/glade/campaign/collections/rda/data/d559000/k...,T2,reference,T2,<NA>,K,1983-10-01,1984-09-30 23:00:00,<NA>,<NA>,0 days 01:00:00


In [13]:
cat_temp.df.head().values

array([['/glade/campaign/collections/rda/data/d559000/kerchunk/wy1980.2d.json',
        'T2', 'reference', 'T2', <NA>, 'K', '1979-10-01',
        '1980-09-30 23:00:00', <NA>, <NA>, '0 days 01:00:00'],
       ['/glade/campaign/collections/rda/data/d559000/kerchunk/wy1981.2d.json',
        'T2', 'reference', 'T2', <NA>, 'K', '1980-10-01',
        '1981-09-30 23:00:00', <NA>, <NA>, '0 days 01:00:00'],
       ['/glade/campaign/collections/rda/data/d559000/kerchunk/wy1982.2d.json',
        'T2', 'reference', 'T2', <NA>, 'K', '1981-10-01',
        '1982-09-30 23:00:00', <NA>, <NA>, '0 days 01:00:00'],
       ['/glade/campaign/collections/rda/data/d559000/kerchunk/wy1983.2d.json',
        'T2', 'reference', 'T2', <NA>, 'K', '1982-10-01',
        '1983-09-30 23:00:00', <NA>, <NA>, '0 days 01:00:00'],
       ['/glade/campaign/collections/rda/data/d559000/kerchunk/wy1984.2d.json',
        'T2', 'reference', 'T2', <NA>, 'K', '1983-10-01',
        '1984-09-30 23:00:00', <NA>, <NA>, '0 days 01:00:0

In [14]:
%%time
test = xr.open_dataset('/gdex/data/d559000/kerchunk/wy1980.2d.json')
test

/glade/u/home/harshah/.conda/envs/osdf/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CPU times: user 48.7 s, sys: 5.18 s, total: 53.9 s
Wall time: 54.2 s


<xarray.Dataset> Size: 11TB
Dimensions:                 (Time: 8784, south_north: 1015, west_east: 1367,
                             soil_layers_stag: 4, snow_layers_stag: 3,
                             west_east_stag: 1368, south_north_stag: 1016,
                             snso_layers_stag: 7)
Coordinates:
  * Time                    (Time) datetime64[ns] 70kB 1979-10-01 ... 1980-09...
    XLAT                    (Time, south_north, west_east) float32 49GB ...
    XLAT_U                  (Time, south_north, west_east_stag) float32 49GB ...
    XLAT_V                  (Time, south_north_stag, west_east) float32 49GB ...
    XLONG                   (Time, south_north, west_east) float32 49GB ...
    XLONG_U                 (Time, south_north, west_east_stag) float32 49GB ...
    XLONG_V                 (Time, south_north_stag, west_east) float32 49GB ...
    XTIME                   (Time) datetime64[ns] 70kB ...
Dimensions without coordinates: south_north, west_east, soil_layers_stag,
                                snow_layers_stag, west_east_stag,
                                south_north_stag, snso_layers_stag
Data variables: (12/194)
    ACDEWC                  (Time, south_north, west_east) float32 49GB ...
    ACDRIPR                 (Time, south_north, west_east) float32 49GB ...
    ACDRIPS                 (Time, south_north, west_east) float32 49GB ...
    ACECAN                  (Time, south_north, west_east) float32 49GB ...
    ACEDIR                  (Time, south_north, west_east) float32 49GB ...
    ACETLSM                 (Time, south_north, west_east) float32 49GB ...
    ...                      ...
    index_snow_layers_stag  (Time, snow_layers_stag) int32 105kB ...
    index_snso_layers_stag  (Time, snso_layers_stag) int32 246kB ...
    index_soil_layers_stag  (Time, soil_layers_stag) int32 141kB ...
    totalIce                (Time, south_north, west_east) float32 49GB ...
    totalLiq                (Time, south_north, west_east) float32 49GB ...
    totalVap                (Time, south_north, west_east) float32 49GB ...
Attributes: (12/151)
    AER_ANGEXP_OPT:                  1
    AER_ANGEXP_VAL:                  1.2999999523162842
    AER_AOD550_OPT:                  1
    AER_AOD550_VAL:                  0.11999999731779099
    AER_ASY_OPT:                     1
    AER_ASY_VAL:                     0.8999999761581421
    ...                              ...
    WEST-EAST_PATCH_END_UNSTAG:      1367
    WEST-EAST_PATCH_START_STAG:      1
    WEST-EAST_PATCH_START_UNSTAG:    1
    W_DAMPING:                       1
    YSU_TOPDOWN_PBLMIX:              0
    history:                         Mon Sep 12 21:16:06 2022: ncks -v XLAT_V...

In [19]:
date = "1980-09-30"
test.T2.sel(Time=date,method='nearest').values

array([[297.47467, 297.46545, 297.45624, ..., 300.5632 , 300.6054 ,
        300.63654],
       [297.47638, 297.51083, 297.50455, ..., 300.54544, 300.5817 ,
        300.59372],
       [297.48605, 297.51703, 297.5169 , ..., 300.52386, 300.55124,
        300.5514 ],
       ...,
       [286.27594, 286.27704, 286.28522, ..., 272.12216, 272.30072,
        271.19397],
       [286.26907, 286.2781 , 286.27634, ..., 271.8276 , 271.85138,
        270.7123 ],
       [286.26056, 286.263  , 286.26395, ..., 270.1154 , 270.14557,
        270.44562]], shape=(1015, 1367), dtype=float32)

- The data is organized in (virtual) zarr stores with one water year's worth of data in one file
- Select a year. This is done by selcting the start time to be Oct 1 of that year or the end time to be Sep 30 of the same year
- This also means that if you want to request data for other days, say Jan 1 for the year YYYY, you first have to load the data for one year i.e., YYYY and then select the data for that particular day. This example is discussed below.


In [13]:
date = "2020-10-01"
# year = "2021"
cat_temp_subset = cat_temp.search(start_time = date)
cat_temp_subset

,unique
path,1
variable,1
format,1
short_name,1
long_name,1
units,1
start_time,1
end_time,1
level,1
level_units,1


### Load data into xarray

In [ ]:
%%time
# Load catalog entries for subset into a dictionary of xarray datasets, and open the first one.
dsets = cat_temp_subset.to_dataset_dict(xarray_open_kwargs={'engine':'kerchunk',"chunks": {}})
#
print(f"\nDataset dictionary keys:\n {dsets.keys()}")


--> The keys in the returned dictionary of datasets are constructed as follows:
	'variable.short_name'


<div><progress max="1" value="0"></progress> 0.00% [0/1 00:00&lt;?]</div>

In [ ]:
# Load the first dataset and display a summary.
dataset_key = list(dsets.keys())[0]
# store_name = dataset_key + ".zarr"
print(dsets.keys())
ds = dsets[dataset_key]
ds = ds.T2
ds

In [ ]:
%%time
desired_time = "2021-01-01T00"
ds.sel(Time=desired_time,method='nearest').plot(cmap='inferno')

In [ ]:
cluster.close()